# Splitify — Multi-stem training (Google Colab)

This notebook trains the **multi-stem** U-Net in `learning/multistem` using the preprocessed HDF5 dataset.

## Prereqs
- Colab runtime with **GPU** enabled (`Runtime → Change runtime type → GPU`).
- `data.zip` uploaded to your Google Drive (should contain `hdf5s/` and `indexes/`).
- The Splitify code in a Git repo you can clone.

## Notes
- We extract data to Colab local disk (`/content`) for faster I/O.
- We regenerate `indexes/*.pkl` on Colab so paths match the Colab filesystem.

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Fill these in
REPO_URL = "<PASTE_YOUR_GIT_REPO_URL_HERE>"  # e.g. https://github.com/you/splitify.git
REPO_DIR = "/content/splitify"

# Path to your uploaded data.zip in Google Drive
DATA_ZIP_IN_DRIVE = "/content/drive/MyDrive/data.zip"


In [ ]:
!rm -rf "{REPO_DIR}"
!git clone "{REPO_URL}" "{REPO_DIR}"
%cd "{REPO_DIR}"


In [ ]:
!python -m pip install -U pip
# Some environments fail to build/install pickle5; fallback install keeps training unblocked.
!pip install -r requirements.txt || pip install torch torchaudio numpy h5py stempeg soundfile tqdm pyyaml librosa numba matplotlib


In [ ]:
# Extract data.zip into the repo root so you get: /content/splitify/hdf5s, /indexes, ...
!test -f "{DATA_ZIP_IN_DRIVE}" && echo "Found data.zip" || (echo "Missing data.zip at: {DATA_ZIP_IN_DRIVE}" && exit 1)
!unzip -q "{DATA_ZIP_IN_DRIVE}" -d "{REPO_DIR}"
!ls -la "{REPO_DIR}"


In [ ]:
# Regenerate indexes on Colab so hdf5 paths match this filesystem
!python preprocessing/index.py \
  --workspace "{REPO_DIR}" \
  --config_yaml "{REPO_DIR}/preprocessing/configs/sr=44100,vocals-bass-drums-other.yaml" \
  --split train


In [ ]:
# Train multi-stem model
# Start with a conservative batch size; increase if you don't hit CUDA OOM.
%cd "{REPO_DIR}/learning/multistem"

!python train.py \
  --workspace "{REPO_DIR}/checkpoints" \
  --index_pkl "{REPO_DIR}/indexes/musdb18/train/sr=44100,vocals-bass-drums-other.pkl" \
  --epochs 20 \
  --batch_size 2 \
  --lr 1e-3


In [ ]:
# Copy checkpoints to Drive so you don't lose them when Colab resets
!mkdir -p /content/drive/MyDrive/splitify_checkpoints
!cp -r "{REPO_DIR}/checkpoints"/* /content/drive/MyDrive/splitify_checkpoints/ || true
!ls -la /content/drive/MyDrive/splitify_checkpoints | tail
